# Target Set Selection

In [1]:
import json, re, tqdm, os, random
import xml.etree.ElementTree as ET
from nltk.probability import FreqDist
path_to_corpus = "/mount/resources/corpora/Royal-Society-Corpus/Royal_Society_Corpus_open_v6.0_texts_vrt/"

## Extracting compounds and their head nouns

In [2]:
def get_sentences_from_file(fname, usePath = True):
    """
    Takes in a (badly) xml-formatted file and returns:
        - the year
        - the sentences as a list of lists of (token, POS, surprisal) str tuples
        using get_sentences_from_text
    """ 
    if usePath:
        fname = os.path.join(path_to_corpus, fname)
    with open(fname) as f:
        text = f.read()
    return get_sentences_from_text(text)

def get_sentences_from_text(text:str):    
    """
    Takes in a (ggf. badly formed) xml-string and returns:
        - the year
        - the sentences as a list of lists of (token, POS, surprisal) str tuples
    """
    text = re.sub(r'</?(page|inferred).*>\n', '', text)
    root = ET.fromstring(text)
    #if root.attrib['isAbstractOf'] != "" or root.attrib['language'] not in ['en', '']:
    if root.attrib['language'] not in ['en', '']: # filter out non-English texts, but include abstracts
        return None
    
    sentences = []
    for sentence in root:
        #assert sentence.tag == 's'
        tokens = ''.join(sentence.itertext()).split('\n')
        tokens = [(row.split()[0], row.split()[1], row.split()[4]) for row in tokens if len(row) > 0]
        sentences.append(tokens)

    return sentences

In [3]:
occ_as_head = {}
def get_heads_from_sents(sentences):
    global occ_as_head
    if not sentences:
        return
    for tokens in sentences:
        for i in range(len(tokens)-1):
            # detect compounds
            if tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
                head = tokens[i+1][0].lower()
                modifier = tokens[i][0].lower()
                if head not in occ_as_head:
                    occ_as_head[head] = []
                occ_as_head[head].append(modifier)
                

for file in tqdm.tqdm(os.listdir(path_to_corpus)):
    sents = get_sentences_from_file(file)
    get_heads_from_sents(sents)

  0%|          | 3/17520 [00:00<12:24, 23.53it/s]

100%|██████████| 17520/17520 [04:05<00:00, 71.38it/s] 


In [6]:
with open("head_occs.json", 'w') as jsonf:
    json.dump(occ_as_head, jsonf)

We take the 100 most common heads and for each the 10 most common modifiers. Filtering for at least 40 occurrences in the open version and being words results in 345 compounds.

In [4]:
#with open('features_v1.json', 'r') as jsonf:
    #occ_as_head = json.load(jsonf)[1]
    #occ_as_mod = json.load(jsonf)[2]

productivities = []
for head in occ_as_head.keys():
    if len(head) > 2:
        productivities.append((head, len(set(occ_as_head[head]))))

most_prod = sorted(productivities, key=lambda x:x[1], reverse=True)[:100]

# for head, _ in most_prod:
#     print(head, "\tmost common modifiers: ", FreqDist(occ_as_head[head]).most_common(10))

targets = []
for head, _ in most_prod:
    line = ""
    fd = FreqDist(occ_as_head[head]).most_common(8)
    for modifier, freq in fd:
        if freq > 50 and re.fullmatch('[a-z]+', head) and re.fullmatch('[a-z]+', modifier):
            targets.append(modifier + " " + head)
            
with open("raw_compound_list.txt", "w") as f:
    for compound in targets:
        f.write(compound + '\n')

I (conservatively) removed words that weren't noun compounds and (very conservatively) some where people would have no chance of knowing what one constituent means (or that seemed to have no definition at all) (examples: baryta water, orbitar process) and ended up with 214

further removing chemical compound terms (only chlorides): 206

Further removed a random 6:
stream function, ammonium salt, temperature difference, directive power, brass wire, spring water

In [1]:
import csv
with open("/home/users1/zabereus/zabereus/master_env/thesis/targets/comp-datasets-release-v2/annotations/filtered/en.filtered.csv", "r") as anno_f:
    all_ratings = []
    for line in csv.reader(anno_f, delimiter='\t'):
        all_ratings.append((line[1], line[6], line[8])) # compound, head, mod
    all_ratings.sort(key=lambda x:x[1])
    print("Heads lowest score: ", all_ratings[:5])
    print("Heads highest score: ", all_ratings[-5:-1])
    all_ratings.sort(key=lambda x:x[2])
    print("Mods lowest score: ", all_ratings[:5])
    print("Mods highest score: ", all_ratings[-5:-1])


Heads lowest score:  [('bad_hat', '0.0000', '4.2000'), ('eager_beaver', '0.0000', '4.3684'), ('old_hat', '0.0625', '3.6875'), ('elbow_grease', '0.1538', '0.9231'), ('sex_bomb', '0.1765', '2.8824')]
Heads highest score:  [('cellular_phone', '5.0000', '3.5333'), ('lime_tree', '5.0000', '4.8750'), ('mental_disorder', '5.0000', '5.0000'), ('milk_tooth', '5.0000', '0.8333')]
Mods lowest score:  [('silver_lining', '0.2222', '0.1667'), ('wet_blanket', '0.2222', '0.2222'), ('nut_case', '0.4375', '0.2500'), ('dead_end', '4.1667', '0.2778'), ('goose_egg', '0.5000', '0.3000')]
Mods highest score:  [('computer_program', '4.7500', '4.9375'), ('health_check', '4.2000', '5.0000'), ('insurance_company', '4.8889', '5.0000'), ('mental_disorder', '5.0000', '5.0000')]


elbow grease
sex bomb
lime tree
mental disorder
milk tooth
silver lining
nut case
insurance company

## Example sentences
[unused in the thesis]

In [11]:
example_sents = {}
with open("../all_example_sents.json", "r") as f:
    all_example_sents = json.load(f)
for compound, sents in all_example_sents.items():
    if not sents:
        print(f"compound {compound} not found")
        continue
    s = random.choice(sents)
    #TODO cleanup?
    s = re.sub(r' ([.,)!?\'])', r'\1', s)
    s = re.sub(r'([(\']) ', r'\1', s)
    example_sents[compound] = s

compound spinode curve not found
compound porcelain tube not found
compound paraffin series not found
compound hydrogen series not found
compound steel point not found
compound zinc plate not found
compound expansion apparatus not found
compound scale value not found
compound haemoglobin value not found
compound flame spectrum not found
compound viscosity work not found
compound arsenic acid not found
compound platina wire not found
compound cross wire not found
compound muscle substance not found
compound spark length not found
compound motor power not found
compound cabinet number not found
compound sunspot area not found
compound spot area not found
compound litmus paper not found
compound turmeric paper not found
compound sewer air not found
compound material change not found
compound brass scale not found
compound platinum salt not found
compound directive force not found
compound injection mass not found
compound investing mass not found
compound penny weight not found
compound i

In [ ]:
for compound, ex in example_sents.items():
    print(compound, "\t\t", ex) 

In [ ]:
def sample_examples(n=3, method="random"):
    with open("../all_example_sents.json", "r") as f:
        all_example_sents = json.load(f) 
        #example_sents = {compound: [] for compound in all_example_sents.keys()}
        for compound in all_example_sents:
            all_sents = all_example_sents[compound]
            if len(all_example_sents[compound]) >= n:
                example_sents[compound] = all_sents.copy()
            elif method == "newest":
                example_sents[compound] = all_sents[-n:]
            elif method == "random":
                example_sents[compound] = random.sample(all_sents, n)
            elif method == "balanced":
                indexes = [i * len(all_sents)/n for i in range(n)]
                example_sents[compound] = [all_sents[i] for i in indexes]
    with open("../example_sents.json", "r") as outfile:
        json.dump(example_sents, outfile)

In [ ]:
f = open("../example_sents.json", "r")
example_sents = json.load(f)
print("Already done: ", len(example_sents))

def get_good_example_sents():
    not_found = []
    with open("../all_example_sents.json", "r") as f:
        all_example_sents = json.load(f)
    for compound, sents in all_example_sents.items():
        random.shuffle(sents)
        judgement = 'n'
        while judgement =='n' and compound not in example_sents:
            if not sents:
                not_found.append(compound)
                break
            s = sents.pop()
            s = re.sub(r' ([.,)!?\]])', r'\1', s)
            s = re.sub(r'([\[(]) ', r'\1', s)
            judgement = input(f"{compound}: {s}")
            #sents.remove(s)
            if judgement.startswith('c'):
                _, seperator, ix = judgement.split()
                s = s.split(seperator)[int(ix)]
            if judgement == 's':
                with open("../example_sents.json", "w") as f:
                    json.dump(example_sents, f)
                    return
            if judgement != 'n':
                example_sents[compound] = s.strip()
    print(f"No example sentences present: {len(not_found)} compounds")
    print(example_sents)
get_good_example_sents()
f.close()

Already done:  40


In [15]:
with open("../example_sents.json", "r") as f:
    example_sents = json.load(f)
    print("Already done: ", len(example_sents))

Already done:  75
